# 对比测试 —— compared1-4 模型在 type1 数据上的复振幅重建

测试 4 个 epoch=200 模型（compared1~4）在 4 个 type1 真实数据集上的表现：
- **数据集**: HSTA_type1, JZX1_type1, MXWKY1_type1, USAF_type1
- **模型**: compared1, compared2, compared3, compared4 (epoch_200)
- **距离**: 取 dist 在 [1.5, 3.0] mm 范围内的最小距离
- **预处理**: JZX1 不做归一化，其余三个数据集对衍射图做 min-max 归一化

结果保存到 `compare_results/` 目录下。

In [ ]:
import sys
sys.path.insert(0, '/root/autodl-tmp/PINet_cpx_cl')

import os
import numpy as np
from scipy.io import loadmat

import torch
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

from src.model import PINet_cpx_v6
from src.config import *
from src.utilities import normalize_tensor

print('Imports OK, device:', DEVICE)

In [ ]:
# ==================== 全局配置 ====================
device = DEVICE

# 4 个模型目录
model_folders = {
    'compared1': os.path.join(SAVE_DIR, 'model_saved_pinet_compared1'),
    'compared2': os.path.join(SAVE_DIR, 'model_saved_pinet_compared2'),
    'compared3': os.path.join(SAVE_DIR, 'model_saved_pinet_compared3'),
    'compared4': os.path.join(SAVE_DIR, 'model_saved_pinet_compared4'),
}
model_filename = 'model_pinet_size256_epoch_200_batchsize4_1.5mm_to_3.0mm.pth'
fold_iters = 9
percent = 0.5  # 前后裁剪百分比

# 4 个 type1 数据集
datasets = ['HSTA_type1', 'JZX1_type1', 'MXWKY1_type1', 'USAF_type1']

# JZX1 不需要对衍射图做归一化
SKIP_NORMALIZE = {'JZX1_type1'}

# 输出目录
output_dir = os.path.join(PROJECT_ROOT, 'compare_results')
os.makedirs(output_dir, exist_ok=True)

print(f'Models: {list(model_folders.keys())}')
print(f'Datasets: {datasets}')
print(f'Output: {output_dir}')
print(f'Skip normalize: {SKIP_NORMALIZE}')

In [ ]:
def load_checkpoint(checkpoint_path, model):
    """加载模型 checkpoint。"""
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])


def compute_TF(Nx, Ny, dx_val, z_m):
    """计算 Angular Spectrum 传递函数。"""
    k_val = 2 * np.pi / WAVELENGTH
    Lx = Nx * dx_val
    Ly = Ny * dx_val
    fx = np.linspace(-1 / (2 * dx_val), 1 / (2 * dx_val) - 1 / Lx, Nx)
    fy = np.linspace(-1 / (2 * dx_val), 1 / (2 * dx_val) - 1 / Ly, Ny)
    FX, FY = np.meshgrid(fy, fx)
    TF = np.exp(1j * z_m * np.sqrt(k_val ** 2 - (2 * np.pi * FX) ** 2 - (2 * np.pi * FY) ** 2))
    return torch.from_numpy(TF).to(device).to(torch.complex64)


def preprocess_diffraction(diffraction, do_normalize):
    """
    衍射图预处理：开根号 -> 可选归一化 -> 对齐 8 的倍数。
    do_normalize=True 时执行 min-max 归一化（JZX1 跳过）。
    """
    H, W = diffraction.shape
    Nx, Ny = H // 8 * 8, W // 8 * 8  # 对齐到 8

    diff_np = np.sqrt(diffraction.astype(np.float64))
    if do_normalize:
        diff_np = (diff_np - diff_np.min()) / (diff_np.max() - diff_np.min())
    diff_np = diff_np[:Nx, :Ny]

    return torch.from_numpy(diff_np).to(device).float(), Nx, Ny


def find_best_distance_idx(dist_array):
    """在 [1.5, 3.0] mm 范围内找到最小距离对应的索引。"""
    dist_flat = dist_array.flatten()
    valid_mask = (dist_flat >= 1.5) & (dist_flat <= 3.0)
    valid_indices = np.where(valid_mask)[0]
    if len(valid_indices) == 0:
        raise ValueError(f'No distance in [1.5, 3.0] mm! Available: {dist_flat}')
    best_idx = valid_indices[np.argmin(dist_flat[valid_indices])]
    return best_idx, dist_flat[best_idx]

In [ ]:
# ==================== 主处理循环 ====================
# results[data_name] = (dist_val, [(model_label, amp, phs), ...])
results = {}

for data_name in datasets:
    print(f'\n{"="*60}')
    print(f'Processing: {data_name}')
    print(f'{"="*60}')

    # ---- 加载数据 ----
    data_path = os.path.join(REAL_DATA_DIR, f'{data_name}_new.mat')
    data = loadmat(data_path)
    print(f'  Loaded: {os.path.basename(data_path)}')

    all_Shift_Samples = data['Shift_Samples'].astype(np.float64)
    dist_array = data['dist']
    print(f'  Shift_Samples: {all_Shift_Samples.shape}, dist: {dist_array.flatten()}')

    # ---- 选择距离 ----
    best_idx, dist_val = find_best_distance_idx(dist_array)
    print(f'  Selected: idx={best_idx}, dist={dist_val:.4f} mm')

    # ---- 衍射图预处理 ----
    diffraction = all_Shift_Samples[:, :, best_idx]
    do_normalize = data_name not in SKIP_NORMALIZE
    print(f'  Normalize: {"YES" if do_normalize else "NO (JZX1)"}')

    diff_tensor, Nx, Ny = preprocess_diffraction(diffraction, do_normalize)
    print(f'  Input tensor: {diff_tensor.shape}, range=[{diff_tensor.min():.3f}, {diff_tensor.max():.3f}]')

    # ---- 计算 TF ----
    dx_val = 1.67e-6
    z_m = dist_val * 1e-3
    TF_torch = compute_TF(Nx, Ny, dx_val, z_m)
    print(f'  TF: shape={TF_torch.shape}, z={z_m*1e3:.3f} mm, dx={dx_val*1e6:.2f} um')

    # ---- 依次加载 4 个模型并推理 ----
    data_results = []
    for model_label, folder in model_folders.items():
        checkpoint_path = os.path.join(folder, model_filename)
        print(f'  [{model_label}] Loading: {os.path.basename(checkpoint_path)}')

        model = PINet_cpx_v6(fold_iters=fold_iters).to(device)
        load_checkpoint(checkpoint_path, model)
        model.eval()

        with torch.no_grad():
            pred, y_rec = model(diff_tensor.unsqueeze(0).unsqueeze(0), TF_torch)

        pred_np = pred.squeeze(0).squeeze(0).detach().cpu().numpy()
        amp = np.abs(pred_np)
        phs = np.angle(pred_np)

        data_results.append((model_label, amp, phs))
        print(f'         amp range=[{amp.min():.3f}, {amp.max():.3f}], phs range=[{phs.min():.3f}, {phs.max():.3f}]')

    results[data_name] = (dist_val, data_results)

In [ ]:
# ==================== 可视化并保存结果 ====================
model_labels = list(model_folders.keys())
n_models = len(model_labels)

for data_name, (dist_val, data_results) in results.items():
    fig, axes = plt.subplots(2, n_models, figsize=(4 * n_models, 8))

    for i, (label, amp, phs) in enumerate(data_results):
        # ---- 振幅 ----
        vmin_amp = np.percentile(amp, percent)
        vmax_amp = np.percentile(amp, 100 - percent)
        amp_disp = np.clip(amp, vmin_amp, vmax_amp)
        amp_disp = (amp_disp - amp_disp.min()) / (amp_disp.max() - amp_disp.min())

        axes[0, i].imshow(amp_disp, cmap='gray')
        axes[0, i].set_title(f'{label}')
        axes[0, i].axis('off')

        # ---- 相位 ----
        vmin_phs = np.percentile(phs, percent)
        vmax_phs = np.percentile(phs, 100 - percent)
        phs_disp = np.clip(phs, vmin_phs, vmax_phs)
        phs_disp = (phs_disp - phs_disp.min()) / (phs_disp.max() - phs_disp.min())

        axes[1, i].imshow(phs_disp, cmap='gray')
        axes[1, i].set_title(f'{label}')
        axes[1, i].axis('off')

    # 添加行标签
    axes[0, 0].set_ylabel('Amplitude', fontsize=12)
    axes[1, 0].set_ylabel('Phase', fontsize=12)

    fig.suptitle(f'{data_name}  |  dist = {dist_val:.4f} mm', fontsize=14)
    plt.tight_layout()

    # 保存
    save_path = os.path.join(output_dir, f'comparison_{data_name}.png')
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved: {save_path}')

    # 同时保存单独的全分辨率 amp/phase 图片
    for label, amp, phs in data_results:
        vmin_amp = np.percentile(amp, percent)
        vmax_amp = np.percentile(amp, 100 - percent)
        amp_save = np.clip(amp, vmin_amp, vmax_amp)
        amp_save = (amp_save - amp_save.min()) / (amp_save.max() - amp_save.min())

        vmin_phs = np.percentile(phs, percent)
        vmax_phs = np.percentile(phs, 100 - percent)
        phs_save = np.clip(phs, vmin_phs, vmax_phs)
        phs_save = (phs_save - phs_save.min()) / (phs_save.max() - phs_save.min())

        amp_path = os.path.join(output_dir, f'amp_{data_name}_{label}.png')
        phs_path = os.path.join(output_dir, f'phs_{data_name}_{label}.png')
        plt.imsave(amp_path, amp_save, cmap='gray')
        plt.imsave(phs_path, phs_save, cmap='gray')

    print(f'  Individual amp/phase images saved.')

print(f'\nAll results saved to: {output_dir}')